# Cost Benchmark - LLaMA-3.1-8B-Instruct
# Measures: Latency, TTFT, GPU Memory, Disk Storage

## Install Packages

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

!pip install -q datasets transformers pandas numpy tqdm google-generativeai requests codecarbon psutil gputil

## Import Libraries

In [2]:
import os
import csv
import time
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from tqdm.auto import tqdm
import psutil
import subprocess

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Mount Drive and Login

In [3]:
drive.mount('/content/drive')

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in successfully")
else:
    print("No HF_TOKEN found")

Mounted at /content/drive
Logged in successfully


## Configuration

In [4]:
# Model config
MODEL_NAME = "LLaMA-3.1-8B-Instruct"
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
LORA_ADAPTER = "Lakshan2003/Llama-3.1-8B-Instruct-customerservice"

# Paths
TEST_DATASET_REPO = "Lakshan2003/slm-cost-benchmark-testset-1000"
RESULTS_DIR = f"/content/drive/MyDrive/fyp-2025/cost_benchmark_results/LLaMA-3.1-8B-Instruct"
RESULTS_CSV = os.path.join(RESULTS_DIR, f"LLaMA-3.1-8B-Instruct_results.csv")
METRICS_CSV = os.path.join(RESULTS_DIR, f"LLaMA-3.1-8B-Instruct_metrics.csv")
OUTPUT_REPO = f"Lakshan2003/LLaMA-3.1-8B-Instruct-cost-benchmark-results"

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Model: LLaMA-3.1-8B-Instruct")
print(f"Results will be saved to: {RESULTS_DIR}")

Model: LLaMA-3.1-8B-Instruct
Results will be saved to: /content/drive/MyDrive/fyp-2025/cost_benchmark_results/LLaMA-3.1-8B-Instruct


## Load Test Dataset

In [5]:
test_dataset = load_dataset(TEST_DATASET_REPO, split="train")
print(f"Loaded {len(test_dataset)} test samples")

README.md:   0%|          | 0.00/582 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 1000 test samples


## GPU Memory Tracking Functions

In [6]:
def get_gpu_memory_mb():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0

def get_model_disk_size_gb(model_path):
    """Calculate disk storage for model files in GB"""
    try:
        result = subprocess.run(
            ["du", "-sb", model_path],
            capture_output=True,
            text=True
        )
        size_bytes = int(result.stdout.split()[0])
        return size_bytes / (1024 ** 3)
    except:
        return 0

## Load Model

In [7]:
print(f"Loading base model: meta-llama/Llama-3.1-8B-Instruct")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading LoRA adapter: Lakshan2003/Llama-3.1-8B-Instruct-customerservice")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
model = model.merge_and_unload()
model.eval()

print("Model loaded successfully")

# Get model disk size
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
MODEL_DISK_SIZE_GB = get_model_disk_size_gb(cache_dir)
print(f"Model disk storage: {MODEL_DISK_SIZE_GB:.2f} GB")

Loading base model: meta-llama/Llama-3.1-8B-Instruct


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading LoRA adapter: Lakshan2003/Llama-3.1-8B-Instruct-customerservice


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Model loaded successfully
Model disk storage: 15.12 GB


## Prompt Template

In [8]:
# Chat template for LLaMA-3.1-8B-Instruct
PROMPT_TEMPLATE = """<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
{instruction}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Conversation History:
{history}

Client Question:
{client_question}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

# Generation config
GENERATION_CONFIG = {
    "max_new_tokens": 128,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 50,
}

def build_prompt(example):
    """Build prompt from example"""
    return PROMPT_TEMPLATE.format(
        instruction=example.get("instruction", ""),
        history=example.get("history_summary", ""),
        client_question=example.get("client_question", "")
    )

## Resume Logic

In [9]:
# Track processed IDs
processed_ids = set()

if os.path.exists(RESULTS_CSV):
    try:
        prev_df = pd.read_csv(RESULTS_CSV)
        if "conversation_id" in prev_df.columns:
            processed_ids = set(prev_df["conversation_id"].astype(str))
            print(f"Resuming from {len(processed_ids)} saved rows")
    except Exception as e:
        print(f"CSV unreadable, starting fresh: {e}")

# Filter unprocessed samples
to_run = []
for ex in test_dataset:
    cid = str(ex.get("conversation_id", ""))
    if cid and cid not in processed_ids:
        to_run.append(ex)

print(f"Pending samples: {len(to_run)}")

Pending samples: 1000


## Inference Loop with Metrics

In [10]:
# Setup
torch.set_grad_enabled(False)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left"

# EOS token IDs
eos_ids = [tokenizer.eos_token_id]
end_token = "<|end|>"
if hasattr(tokenizer, "get_vocab") and end_token in tokenizer.get_vocab():
    end_id = tokenizer.convert_tokens_to_ids(end_token)
    if end_id != tokenizer.eos_token_id:
        eos_ids = [tokenizer.eos_token_id, end_id]

RESULT_COLS = [
    "conversation_id",
    "instruction",
    "history",
    "history_summary",
    "client_question",
    "ground_truth",
    "generated_answer",
]

METRIC_COLS = [
    "conversation_id",
    "latency_seconds",
    "ttft_seconds",
    "gpu_memory_mb",
    "disk_storage_gb",
]

pbar = tqdm(total=len(to_run), desc="Generating", unit="sample")

for ex in to_run:
    conversation_id = str(ex.get("conversation_id", ""))
    instruction = ex.get("instruction", "")
    history = ex.get("history", "")
    history_summary = ex.get("history_summary", "")
    client_question = ex.get("client_question", "")
    ground_truth = ex.get("ground_truth", "")

    input_prompt = build_prompt(ex)

    # TOKENIZATION
    enc = tokenizer(
        [input_prompt],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length,
    ).to(model.device)

    torch.cuda.reset_peak_memory_stats()
    gpu_mem_before = get_gpu_memory_mb()

    # GENERATE FIRST TOKEN (TTFT)
    torch.cuda.synchronize()
    start_ttft = time.time()

    with torch.no_grad():
        first_token_out = model.generate(
            **enc,
            max_new_tokens=1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=eos_ids,
            return_dict_in_generate=True,
            output_scores=False,
            use_cache=True,
        )

    torch.cuda.synchronize()
    ttft = time.time() - start_ttft

    # GENERATE REMAINING TOKENS (FULL LATENCY)
    torch.cuda.synchronize()
    start_latency = time.time()

    with torch.no_grad():
        full_out = model.generate(
            **enc,
            **GENERATION_CONFIG,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=eos_ids,
            return_dict_in_generate=True,
            use_cache=True,
        )

    torch.cuda.synchronize()
    latency = time.time() - start_latency

    # MEMORY
    gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    # DECODE
    seqs = full_out.sequences
    start_idx = enc.input_ids.shape[1]
    generated_text = tokenizer.decode(
        seqs[0, start_idx:],
        skip_special_tokens=True
    ).strip()

    # SAVE RESULTS
    result_row = {
        "conversation_id": conversation_id,
        "instruction": instruction,
        "history": history,
        "history_summary": history_summary,
        "client_question": client_question,
        "ground_truth": ground_truth,
        "generated_answer": generated_text,
    }

    metrics_row = {
        "conversation_id": conversation_id,
        "latency_seconds": latency,
        "ttft_seconds": ttft,
        "gpu_memory_mb": gpu_memory_mb,
        "disk_storage_gb": MODEL_DISK_SIZE_GB,
    }

    pd.DataFrame([result_row], columns=RESULT_COLS).to_csv(
        RESULTS_CSV,
        mode="a",
        header=not os.path.exists(RESULTS_CSV),
        index=False,
        quoting=csv.QUOTE_MINIMAL,
    )

    pd.DataFrame([metrics_row], columns=METRIC_COLS).to_csv(
        METRICS_CSV,
        mode="a",
        header=not os.path.exists(METRICS_CSV),
        index=False,
        quoting=csv.QUOTE_MINIMAL,
    )

    pbar.update(1)

pbar.close()
print("\nInference completed!")

Generating:   0%|          | 0/1000 [00:00<?, ?sample/s]


Inference completed!


## Calculate Summary Statistics

In [11]:
# Load metrics
metrics_df = pd.read_csv(METRICS_CSV)

print("\n=== Summary Statistics ===")
print(f"\nTotal Samples: {len(metrics_df)}")
print(f"\nLatency (seconds):")
print(f"  Mean: {metrics_df['latency_seconds'].mean():.4f}")
print(f"  Median: {metrics_df['latency_seconds'].median():.4f}")
print(f"  Std: {metrics_df['latency_seconds'].std():.4f}")
print(f"  Min: {metrics_df['latency_seconds'].min():.4f}")
print(f"  Max: {metrics_df['latency_seconds'].max():.4f}")

print(f"\nTTFT (seconds):")
print(f"  Mean: {metrics_df['ttft_seconds'].mean():.4f}")
print(f"  Median: {metrics_df['ttft_seconds'].median():.4f}")
print(f"  Std: {metrics_df['ttft_seconds'].std():.4f}")

print(f"\nGPU Memory (MB):")
print(f"  Mean: {metrics_df['gpu_memory_mb'].mean():.2f}")
print(f"  Max: {metrics_df['gpu_memory_mb'].max():.2f}")

print(f"\nDisk Storage (GB): {MODEL_DISK_SIZE_GB:.2f}")

# Save summary
summary_path = os.path.join(RESULTS_DIR, f"LLaMA-3.1-8B-Instruct_summary.txt")
with open(summary_path, "w") as f:
    f.write(f"Model: LLaMA-3.1-8B-Instruct\n")
    f.write(f"Total Samples: {len(metrics_df)}\n\n")
    f.write(f"Latency (seconds):\n")
    f.write(f"  Mean: {metrics_df['latency_seconds'].mean():.4f}\n")
    f.write(f"  Median: {metrics_df['latency_seconds'].median():.4f}\n")
    f.write(f"  Std: {metrics_df['latency_seconds'].std():.4f}\n")
    f.write(f"\nTTFT (seconds):\n")
    f.write(f"  Mean: {metrics_df['ttft_seconds'].mean():.4f}\n")
    f.write(f"  Median: {metrics_df['ttft_seconds'].median():.4f}\n")
    f.write(f"\nGPU Memory (MB):\n")
    f.write(f"  Mean: {metrics_df['gpu_memory_mb'].mean():.2f}\n")
    f.write(f"  Max: {metrics_df['gpu_memory_mb'].max():.2f}\n")
    f.write(f"\nDisk Storage (GB): {MODEL_DISK_SIZE_GB:.2f}\n")

print(f"\nSummary saved to: {summary_path}")


=== Summary Statistics ===

Total Samples: 1000

Latency (seconds):
  Mean: 1.8055
  Median: 1.7745
  Std: 0.5373
  Min: 0.4957
  Max: 3.6793

TTFT (seconds):
  Mean: 0.0431
  Median: 0.0416
  Std: 0.0340

GPU Memory (MB):
  Mean: 15466.52
  Max: 15507.10

Disk Storage (GB): 15.12

Summary saved to: /content/drive/MyDrive/fyp-2025/cost_benchmark_results/LLaMA-3.1-8B-Instruct/LLaMA-3.1-8B-Instruct_summary.txt


## Push to HuggingFace

In [12]:
from datasets import Dataset

# Load both CSVs
results_df = pd.read_csv(RESULTS_CSV)
metrics_df = pd.read_csv(METRICS_CSV)

# Merge on conversation_id
final_df = results_df.merge(metrics_df, on="conversation_id", how="inner")

# Save combined version locally
FINAL_CSV = os.path.join(RESULTS_DIR, "combined_results_metrics.csv")
final_df.to_csv(FINAL_CSV, index=False)

print(f"Combined file saved: {FINAL_CSV}")

Combined file saved: /content/drive/MyDrive/fyp-2025/cost_benchmark_results/LLaMA-3.1-8B-Instruct/combined_results_metrics.csv


In [13]:
final_dataset = Dataset.from_pandas(final_df)

final_dataset.push_to_hub(OUTPUT_REPO, private=True)

print(f"Pushed combined dataset to: {OUTPUT_REPO}")


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  88%|########7 | 1.02MB / 1.16MB            

README.md:   0%|          | 0.00/710 [00:00<?, ?B/s]

Pushed combined dataset to: Lakshan2003/LLaMA-3.1-8B-Instruct-cost-benchmark-results
